# Upload Generated Recipes To Webapp

Bulk-upload generated recipe JSONs from `data/recipes_generated/` to the webapp storage backend (GCS via `src.gcs_storage`).

This notebook also uploads the linked card image (if the JSON points to a local `/images/recipes/...` file) and rewrites `card image file` in the uploaded JSON to the webapp `/media/...` route.

Recommended flow:
1. Run setup/config cells
2. Run preview cell
3. Run dry-run upload
4. Set `UPLOAD_DRY_RUN = False` and run the upload cell


In [1]:
from __future__ import annotations

import json
import mimetypes
import os
import sys
import time
from pathlib import Path
from typing import Any


def _find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (root / 'src').exists() and (root / 'data').exists():
            return root
    raise RuntimeError('Could not find project root (expected `src/` and `data/` folders).')


def _load_simple_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if not key:
            continue
        os.environ.setdefault(key, value)


def _normalize_path_env(var_name: str, project_root: Path) -> None:
    value = (os.getenv(var_name) or '').strip()
    if not value:
        return
    path = Path(value)
    if not path.is_absolute():
        path = (project_root / path).resolve()
    os.environ[var_name] = str(path)


def _pick_existing_service_account_key(project_root: Path) -> str | None:
    candidates = [
        project_root / 'secrets' / 'api_bucket_db_key.json',
        project_root / 'secrets' / 'recetas-webapp-prod-sa.json',
        project_root / 'secrets' / 'deployment-recetas-webapp-sa-prod.json',
    ]
    candidates.extend(sorted((project_root / 'secrets').glob('*.json')))
    seen: set[str] = set()
    for candidate in candidates:
        s = str(candidate.resolve())
        if s in seen:
            continue
        seen.add(s)
        if candidate.exists() and candidate.is_file():
            return s
    return None


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

# Make storage key path resolution deterministic for notebooks (avoids ADC fallback when running from `notebooks/`).
os.chdir(PROJECT_ROOT)

# Load local env files (non-destructive; existing env vars win).
_load_simple_env_file(PROJECT_ROOT / 'env.prod')
_load_simple_env_file(PROJECT_ROOT / 'secrets' / 'env.prod')

# Normalize credentials paths to absolute paths before importing src.gcs_storage.
for env_var in ('API_BUCKET_KEY_FILE', 'GOOGLE_APPLICATION_CREDENTIALS', 'SERVICE_ACCOUNT_KEY_FILE'):
    _normalize_path_env(env_var, PROJECT_ROOT)

# If API_BUCKET_KEY_FILE is unset or missing, derive it from known keys / secrets/*.json.
api_bucket_key = (os.getenv('API_BUCKET_KEY_FILE') or '').strip()
if not api_bucket_key or not Path(api_bucket_key).exists():
    discovered_key = _pick_existing_service_account_key(PROJECT_ROOT)
    if discovered_key:
        os.environ['API_BUCKET_KEY_FILE'] = discovered_key
        api_bucket_key = discovered_key

# If GOOGLE_APPLICATION_CREDENTIALS is unset, mirror API_BUCKET_KEY_FILE for compatibility.
if (not os.getenv('GOOGLE_APPLICATION_CREDENTIALS')) and api_bucket_key:
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = api_bucket_key

print('CWD =', Path.cwd())
print('RECETAS_BUCKET_NAME =', os.getenv('RECETAS_BUCKET_NAME') or os.getenv('BUCKET_NAME') or '')
print('API_BUCKET_KEY_FILE =', os.getenv('API_BUCKET_KEY_FILE', ''))
print('API key file exists =', Path(os.getenv('API_BUCKET_KEY_FILE', '')).exists() if os.getenv('API_BUCKET_KEY_FILE') else False)
print('GOOGLE_APPLICATION_CREDENTIALS =', os.getenv('GOOGLE_APPLICATION_CREDENTIALS', ''))
print('GAC file exists =', Path(os.getenv('GOOGLE_APPLICATION_CREDENTIALS', '')).exists() if os.getenv('GOOGLE_APPLICATION_CREDENTIALS') else False)



PROJECT_ROOT = /home/rafael/E/Python_Projects/Recetas_webapp
CWD = /home/rafael/E/Python_Projects/Recetas_webapp
RECETAS_BUCKET_NAME = recetas-bucket-prod
API_BUCKET_KEY_FILE = /home/rafael/E/Python_Projects/Recetas_webapp/secrets/recetas-webapp-prod-sa.json
API key file exists = True
GOOGLE_APPLICATION_CREDENTIALS = /home/rafael/E/Python_Projects/Recetas_webapp/secrets/recetas-webapp-prod-sa.json
GAC file exists = True


In [2]:
import importlib
import src.gcs_storage as _gcs_storage
from src.pages.recetas_list.core_the_list import RECIPES_INDEX_BLOB as RECIPE_INDEX_BLOB, _rebuild_recipe_index_from_bucket

# Force reload so DEFAULT_KEYFILE is rebuilt from current env if this notebook was re-run.
_gcs_storage = importlib.reload(_gcs_storage)

upload_bytes = _gcs_storage.upload_bytes
media_path = _gcs_storage.media_path
blob_exists = _gcs_storage.blob_exists
bucket_name = _gcs_storage.bucket_name

# These match the webapp defaults unless overridden by env vars.
RECIPES_PREFIX = (os.getenv('RECETAS_RECIPES_PREFIX') or 'recipes').strip('/ ')
IMAGES_PREFIX = (os.getenv('RECETAS_IMAGES_PREFIX') or 'recipes/images').strip('/ ')

GENERATED_JSON_DIR = PROJECT_ROOT / 'data' / 'recipes_generated'
SOURCE_IMAGES_DIR = PROJECT_ROOT / 'images' / 'recipes'
UPLOAD_REPORT_PATH = GENERATED_JSON_DIR / '_upload_report_to_webapp.json'

assert GENERATED_JSON_DIR.exists(), f'Missing generated JSON dir: {GENERATED_JSON_DIR}'
print('Bucket =', bucket_name())
print('DEFAULT_KEYFILE (gcs_storage) =', _gcs_storage.DEFAULT_KEYFILE)
print('gcs_storage key exists =', Path(_gcs_storage.DEFAULT_KEYFILE).exists() if _gcs_storage.DEFAULT_KEYFILE else False)
print('RECIPES_PREFIX =', RECIPES_PREFIX)
print('IMAGES_PREFIX =', IMAGES_PREFIX)
print('RECIPE_INDEX_BLOB =', RECIPE_INDEX_BLOB)
print('GENERATED_JSON_DIR =', GENERATED_JSON_DIR)



Bucket = recetas-bucket-prod
DEFAULT_KEYFILE (gcs_storage) = /home/rafael/E/Python_Projects/Recetas_webapp/secrets/recetas-webapp-prod-sa.json
gcs_storage key exists = True
RECIPES_PREFIX = recipes
IMAGES_PREFIX = recipes/images
GENERATED_JSON_DIR = /home/rafael/E/Python_Projects/Recetas_webapp/data/recipes_generated


In [3]:
# Upload settings (safe defaults)
UPLOAD_DRY_RUN = False               # Preview only. Set False to upload.
MAX_FILES: int | None = None        # e.g. 25 for a test batch
SKIP_EXISTING_RECIPES = True        # Skip recipe JSON if remote recipe blob already exists
SKIP_EXISTING_IMAGES = True         # Skip image upload if remote image blob already exists
FAIL_IF_LINKED_IMAGE_MISSING = True # Error if JSON references a local image that is missing
WRITE_UPLOAD_REPORT = True          # Writes _upload_report_to_webapp.json
SHOW_FIRST_ROWS = 2


In [4]:
def _iter_generated_recipe_jsons(json_dir: Path = GENERATED_JSON_DIR) -> list[Path]:
    files = sorted(p for p in json_dir.glob('*.json') if p.name not in {'_run_report.json', '_upload_report_to_webapp.json'})
    return files


def _guess_content_type(path: Path) -> str:
    guessed, _ = mimetypes.guess_type(str(path))
    return guessed or 'application/octet-stream'


def _recipe_blob_name(slug: str) -> str:
    return f"{RECIPES_PREFIX}/{slug}.json"


def _image_blob_name(slug: str, local_image_path: Path) -> str:
    # Keep image filename, group under the recipe slug.
    return f"{IMAGES_PREFIX}/{slug}/{local_image_path.name}"


def _load_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(payload, dict):
        raise ValueError('JSON root is not an object')
    return payload


def _resolve_local_card_image_path(payload: dict[str, Any]) -> tuple[Path | None, str, str]:
    """Return (local_path_or_none, original_route, note)."""
    route = str(payload.get('card image file') or payload.get('card_image_file') or '').strip()
    if not route:
        return None, '', 'no-image-route'

    # Already webapp/GCS media route; nothing to upload locally unless you want to rehost.
    if route.startswith('/media/'):
        return None, route, 'already-media-route'

    route_clean = route.lstrip('/')
    local_candidate = PROJECT_ROOT / route_clean
    if local_candidate.exists() and local_candidate.is_file():
        return local_candidate, route, 'local-image-found'

    # Fallback for some exports that may store only the filename.
    filename = Path(route).name
    if filename:
        by_name = SOURCE_IMAGES_DIR / filename
        if by_name.exists() and by_name.is_file():
            return by_name, route, 'local-image-found-by-name'

    return None, route, 'linked-image-missing'


def _upsert_tagless_payload_copy(payload: dict[str, Any]) -> dict[str, Any]:
    # Shallow copy is enough for route rewrite + upload serialization.
    return dict(payload)


def upload_one_generated_recipe(
    json_path: Path,
    *,
    dry_run: bool = True,
    skip_existing_recipes: bool = True,
    skip_existing_images: bool = True,
    fail_if_linked_image_missing: bool = True,
) -> dict[str, Any]:
    started = time.time()
    slug = json_path.stem
    recipe_blob = _recipe_blob_name(slug)

    row: dict[str, Any] = {
        'slug': slug,
        'json': str(json_path.relative_to(PROJECT_ROOT)),
        'status': 'pending',
        'recipe_blob': recipe_blob,
        'image_blob': '',
        'image_route_uploaded': '',
        'image_note': '',
        'error': '',
        'ms': 0,
    }

    try:
        payload = _load_json_object(json_path)
        payload_to_upload = _upsert_tagless_payload_copy(payload)

        local_image_path, original_image_route, image_note = _resolve_local_card_image_path(payload_to_upload)
        row['image_note'] = image_note

        if not dry_run and skip_existing_recipes and blob_exists(recipe_blob):
            row['status'] = 'skipped_recipe_exists'
            return row

        if local_image_path is not None:
            image_blob = _image_blob_name(slug, local_image_path)
            row['image_blob'] = image_blob
            row['local_image'] = str(local_image_path.relative_to(PROJECT_ROOT))
            row['image_route_uploaded'] = media_path(image_blob)

            if not dry_run:
                if not (skip_existing_images and blob_exists(image_blob)):
                    upload_bytes(
                        local_image_path.read_bytes(),
                        image_blob,
                        content_type=_guess_content_type(local_image_path),
                        cache_seconds=31536000,
                    )
                payload_to_upload['card image file'] = media_path(image_blob)
            else:
                # Preview the route rewrite that will happen on upload.
                payload_to_upload['card image file'] = media_path(image_blob)

        elif original_image_route and image_note == 'linked-image-missing':
            row['missing_linked_image_route'] = original_image_route
            if fail_if_linked_image_missing:
                raise FileNotFoundError(f'Linked image not found locally for route: {original_image_route}')

        body = (json.dumps(payload_to_upload, ensure_ascii=False, indent=2) + '\n').encode('utf-8')
        row['bytes_json'] = len(body)

        if not dry_run:
            upload_bytes(
                body,
                recipe_blob,
                content_type='application/json; charset=utf-8',
                cache_seconds=0,
            )
            row['status'] = 'uploaded'
        else:
            row['status'] = 'dry_run'

        return row
    except Exception as exc:  # noqa: BLE001
        row['status'] = 'error'
        row['error'] = f'{exc.__class__.__name__}: {exc}'
        return row
    finally:
        row['ms'] = int(round((time.time() - started) * 1000))


def upload_generated_recipes_batch(
    *,
    dry_run: bool = True,
    max_files: int | None = None,
    skip_existing_recipes: bool = True,
    skip_existing_images: bool = True,
    fail_if_linked_image_missing: bool = True,
    write_report: bool = True,
    rebuild_recipe_index_after_upload: bool = True,
) -> list[dict[str, Any]]:
    files = _iter_generated_recipe_jsons()
    if max_files is not None:
        files = files[:max_files]

    rows: list[dict[str, Any]] = []
    for json_path in files:
        row = upload_one_generated_recipe(
            json_path,
            dry_run=dry_run,
            skip_existing_recipes=skip_existing_recipes,
            skip_existing_images=skip_existing_images,
            fail_if_linked_image_missing=fail_if_linked_image_missing,
        )
        rows.append(row)

    if write_report:
        UPLOAD_REPORT_PATH.write_text(json.dumps(rows, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    if (
        (not dry_run)
        and rebuild_recipe_index_after_upload
        and any(str(row.get('status') or '') == 'uploaded' for row in rows)
    ):
        rebuild_started = time.time()
        try:
            print(f'Rebuilding recipe index CSV in bucket: {RECIPE_INDEX_BLOB}')
            rebuilt_count = len(_rebuild_recipe_index_from_bucket())
            print(f'Recipe index rebuilt with {rebuilt_count} rows in {time.time() - rebuild_started:.1f}s')
        except Exception as exc:  # noqa: BLE001
            print(f'WARNING: Failed to rebuild recipe index CSV: {exc.__class__.__name__}: {exc}')

    return rows


def _summarize_rows(rows: list[dict[str, Any]]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for row in rows:
        status = str(row.get('status') or 'unknown')
        counts[status] = counts.get(status, 0) + 1
    return dict(sorted(counts.items()))


def _print_rows(rows: list[dict[str, Any]], limit: int = 20) -> None:
    for row in rows[:limit]:
        print(f"- [{row.get('status')}] {row.get('json')} -> {row.get('recipe_blob')}")
        if row.get('image_blob'):
            print(f"    image: {row.get('image_blob')}  route: {row.get('image_route_uploaded')}")
        if row.get('error'):
            print(f"    error: {row.get('error')}")
    if len(rows) > limit:
        print(f"... and {len(rows) - limit} more")


In [5]:
# Preview what will be uploaded (local generated JSONs only)
preview_files = _iter_generated_recipe_jsons()
print('Generated JSON files found:', len(preview_files))
for p in preview_files[:10]:
    print('-', p.relative_to(PROJECT_ROOT))
if len(preview_files) > 10:
    print('... and', len(preview_files) - 10, 'more')


Generated JSON files found: 7778
- data/recipes_generated/1000-recetas-de-oro-karlos-arguinano-a2527f2a.json
- data/recipes_generated/250-chocolate-chunk-cookies-r89701-b88bd210.json
- data/recipes_generated/3-recetas-quiche-de-internet-6a589db0.json
- data/recipes_generated/a-queimada-330b66f6.json
- data/recipes_generated/acaray-s-de-camarones-con-mojo-y-vinagreta-de-tomates-verdes-r340636-46392e58.json
- data/recipes_generated/aceite-arom-tico-con-aceitunas-negras-r539471-d13ad11c.json
- data/recipes_generated/aceite-con-tomates-secos-romero-y-tomillo-r95296-6b557381.json
- data/recipes_generated/aceite-de-albahaca-r16935-d1b812af.json
- data/recipes_generated/aceites-aromatizados-de-especias-y-hierbas-r71105-6cf20671.json
- data/recipes_generated/aceites-aromatizados-de-larga-duraci-n-r73185-de7559a9.json
... and 7768 more


In [8]:
# Dry-run upload preview (recommended first)
rows_upload = upload_generated_recipes_batch(
    dry_run=UPLOAD_DRY_RUN,
    max_files=MAX_FILES,
    skip_existing_recipes=SKIP_EXISTING_RECIPES,
    skip_existing_images=SKIP_EXISTING_IMAGES,
    fail_if_linked_image_missing=FAIL_IF_LINKED_IMAGE_MISSING,
    write_report=WRITE_UPLOAD_REPORT,
)

print('Mode:', 'DRY_RUN' if UPLOAD_DRY_RUN else 'UPLOAD')
print('Rows processed:', len(rows_upload))
print('Status summary:', _summarize_rows(rows_upload))
print('Report:', UPLOAD_REPORT_PATH.relative_to(PROJECT_ROOT))
_print_rows(rows_upload, limit=SHOW_FIRST_ROWS)


Mode: UPLOAD
Rows processed: 7778
Status summary: {'skipped_recipe_exists': 10, 'uploaded': 7768}
Report: data/recipes_generated/_upload_report_to_webapp.json
- [skipped_recipe_exists] data/recipes_generated/1000-recetas-de-oro-karlos-arguinano-a2527f2a.json -> recipes/1000-recetas-de-oro-karlos-arguinano-a2527f2a.json
- [skipped_recipe_exists] data/recipes_generated/250-chocolate-chunk-cookies-r89701-b88bd210.json -> recipes/250-chocolate-chunk-cookies-r89701-b88bd210.json
... and 7776 more


In [ ]:
# Actual upload (run only when ready)
# 1) Set UPLOAD_DRY_RUN = False in the settings cell.
# 2) Re-run this cell.
#
# rows_upload = upload_generated_recipes_batch(
#     dry_run=UPLOAD_DRY_RUN,
#     max_files=MAX_FILES,
#     skip_existing_recipes=SKIP_EXISTING_RECIPES,
#     skip_existing_images=SKIP_EXISTING_IMAGES,
#     fail_if_linked_image_missing=FAIL_IF_LINKED_IMAGE_MISSING,
#     write_report=WRITE_UPLOAD_REPORT,
# )
# print('Mode:', 'DRY_RUN' if UPLOAD_DRY_RUN else 'UPLOAD')
# print('Rows processed:', len(rows_upload))
# print('Status summary:', _summarize_rows(rows_upload))
# _print_rows(rows_upload, limit=SHOW_FIRST_ROWS)


In [ ]:
# Optional: show only errors from the latest upload report
if UPLOAD_REPORT_PATH.exists():
    latest_rows = json.loads(UPLOAD_REPORT_PATH.read_text(encoding='utf-8'))
    error_rows = [row for row in latest_rows if str(row.get('status')) == 'error']
    print('Errors:', len(error_rows))
    _print_rows(error_rows, limit=max(SHOW_FIRST_ROWS, 50))
else:
    print('No upload report found yet:', UPLOAD_REPORT_PATH)
